# 03 — Model Training
Trains the five competing models. The three supervised models (Logistic
Regression, Random Forest, XGBoost) are each trained under **three
imbalance-handling experiments** (Experiment A: none, B: class weighting,
C: SMOTE), matching the methodology's imbalance-as-experiment design.
The two unsupervised models (Isolation Forest, Autoencoder) are trained
once each, since they do not use labels during fitting.


In [1]:
# ---------------------------------------------------------------------------
# Make the src/ package importable from inside notebooks/, and fix all
# random seeds so this notebook is reproducible (Section 14 requirement).
# ---------------------------------------------------------------------------
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import preprocessing, features, models, evaluation, explainability

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (8, 4)


In [2]:
import joblib

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze("columns")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze("columns")

print(X_train.shape, X_test.shape)


(226980, 31) (56746, 31)


In [3]:
# ---------------------------------------------------------------------------
# Experiments A / B / C: train the supervised models under each imbalance
# strategy and keep every trained model for later comparison.
# ---------------------------------------------------------------------------
trained_supervised = {}
for strategy in models.IMBALANCE_STRATEGIES:
    trained_supervised[strategy] = models.train_supervised_models(X_train, y_train, strategy)
    print(f"Trained supervised models under strategy = '{strategy}'")


Trained supervised models under strategy = 'none'
Trained supervised models under strategy = 'class_weight'
Trained supervised models under strategy = 'smote'


In [4]:
# ---------------------------------------------------------------------------
# Unsupervised models (label-free)
# ---------------------------------------------------------------------------
iso_forest = models.train_isolation_forest(X_train, y_train)
autoencoder = models.train_autoencoder(X_train, y_train)
print("Trained Isolation Forest and Autoencoder.")


Trained Isolation Forest and Autoencoder.


In [5]:
# ---------------------------------------------------------------------------
# Persist every trained model to models/ for reuse by notebooks 04-05 and
# the dashboard, without needing to retrain.
# ---------------------------------------------------------------------------
os.makedirs("../models", exist_ok=True)
for strategy, model_dict in trained_supervised.items():
    for model_name, model_obj in model_dict.items():
        joblib.dump(model_obj, f"../models/{model_name}_{strategy}.joblib")

joblib.dump(iso_forest, "../models/isolation_forest.joblib")
joblib.dump(autoencoder, "../models/autoencoder.joblib")
print("All models saved to models/.")


All models saved to models/.
